# Barcelona Lden models — training

Trains the Barcelona classification + regression models on the **native Lden** target
(`results/bcn_lden.csv`, extracted from `TOTAL_DEN`), using the same 18 features and the same model
definitions as the original `noise_day` study (`07_SL_save_model_*`). Classification target = Lden binned
into the usual 0–4 classes (`<40 … ≥70`). Saved as `*_lden.pkl` pickles for the all-cities transfer test.

## Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle, os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)
import xgboost as xgb

## Load Barcelona features + native Lden target

In [2]:
with open('../../../notebooks/_elena/models/feature_columns.pkl','rb') as f:
    FEATURES = pickle.load(f)
print(len(FEATURES), 'features')

def classify_noise(v):
    if v < 40: return 0
    if v < 50: return 1
    if v < 60: return 2
    if v < 70: return 3
    return 4

data = pd.read_csv('../../../notebooks/_elena/data/bcn_noise_regre_ml_dataset.csv').dropna()
data['road_id'] = data['road_id'].astype(str)
lden = pd.read_csv('../results/bcn_lden.csv'); lden['road_id'] = lden['road_id'].astype(str)
data = data.merge(lden, on='road_id', how='inner')
data['lden_class'] = data['lden_db'].apply(classify_noise)
print('rows:', len(data))
print('Lden class distribution:')
print(data['lden_class'].value_counts(normalize=True).sort_index().round(3))

18 features
rows: 12854
Lden class distribution:
lden_class
0    0.008
1    0.033
2    0.230
3    0.541
4    0.187
Name: proportion, dtype: float64


## Scale + split (same 18 features, test_size=0.2, random_state=42)

In [3]:
MODELS = '../models'; os.makedirs(MODELS, exist_ok=True)
scaler = StandardScaler().fit(data[FEATURES])
X = scaler.transform(data[FEATURES])
with open(f'{MODELS}/Sscaler_lden.pkl','wb') as f: pickle.dump(scaler, f)
with open(f'{MODELS}/feature_columns_lden.pkl','wb') as f: pickle.dump(list(FEATURES), f)

yc = data['lden_class'].to_numpy()
yr = data['lden_db'].to_numpy(dtype=float)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, yc, test_size=0.2, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, yr, test_size=0.2, random_state=42)

## Classification (Lden class)

In [4]:
rows = []
for name, model, fn in [('Logistic Regression', LogisticRegression(max_iter=2000), 'logreg_class_lden.pkl'),
                        ('XGBoost', xgb.XGBClassifier(), 'xgb_class_lden.pkl'),
                        ('Random Forest', RandomForestClassifier(random_state=42), 'rf_class_lden.pkl')]:
    model.fit(Xc_tr, yc_tr); yp = model.predict(Xc_te)
    with open(f'{MODELS}/{fn}','wb') as f: pickle.dump(model, f)
    rows.append({'model':name, 'accuracy':accuracy_score(yc_te,yp),
                 'macro_f1':f1_score(yc_te,yp,average='macro'),
                 'within_1':np.mean(np.abs(yp-yc_te)<=1)})
pd.DataFrame(rows).set_index('model').round(4)

,accuracy,macro_f1,within_1
model,,,
Logistic Regression,0.6138,0.4460,0.9864
XGBoost,0.7480,0.7590,0.9942
Random Forest,0.7635,0.7582,0.9942


## Regression (Lden dB)

In [5]:
rows = []
for name, model, fn in [('Linear Regression', LinearRegression(), 'linreg_regre_lden.pkl'),
                        ('XGBoost', xgb.XGBRegressor(), 'xgb_regre_lden.pkl'),
                        ('Random Forest', RandomForestRegressor(random_state=42), 'rf_regre_lden.pkl')]:
    model.fit(Xr_tr, yr_tr); yp = model.predict(Xr_te)
    with open(f'{MODELS}/{fn}','wb') as f: pickle.dump(model, f)
    rows.append({'model':name, 'r2':r2_score(yr_te,yp), 'mae':mean_absolute_error(yr_te,yp),
                 'rmse':np.sqrt(mean_squared_error(yr_te,yp))})
pd.DataFrame(rows).set_index('model').round(4)

,r2,mae,rmse
model,,,
Linear Regression,0.4529,4.0354,5.1362
XGBoost,0.6787,2.9756,3.9363
Random Forest,0.7081,2.8002,3.7520


Barcelona held-out baseline for the Lden target. Compare to the `noise_day` study (RF 0.749 acc /
0.704 R²): Lden is a slightly different, louder target, so the numbers shift but should be of similar
magnitude (same features). Saved pickles drive `02_test_all_cities_lden.ipynb`.